# Inside a Molecular Dynamics Code
## (Adapted from code available at https://github.com/m-rivera/ljmd)
## Lennard-Jones molecular dynamics

This program simulates the classical dynamics of Argon atoms where the interatomic interactions are mediated by a Lennard-Jones interaction of potential and force:

$V_{LJ}(r) = 4 \epsilon [(\frac{\sigma}{r})^{12} - (\frac{\sigma}{r})^6]$

$|F_{LJ}(r)| = 48 \epsilon [ \frac{\sigma^{12}}{r^{13}} - \frac{\sigma^{6}}{2r^{7}}]$

Where $\epsilon$ is the depth of the potential well and $\sigma$ is the interatomic distance at which the interaction is zero.

## Program specifications
The simulation involves:

- Initial coordinates at FCC positions.
- Periodic boundary conditions with the minimum-image criterion.
- Time integration using the [Velocity Verlet Algorithm](https://en.wikipedia.org/wiki/Verlet_integration#Velocity_Verlet).
- A velocity rescaling thermostat.

## Outputs
The program returns the following files:
- `trajectory.xyz`
- `rdf.csv`
- `report.csv`

## Function definitions

The necessary functions are defined for the code to run:
- `pos_to_xyz()`: Writes out atomic positions to a file in the format "Ar x y z".
- `interatomic_potential()`: Returns the Lennard-Jones energy associated with a given interatomic distance in Angstrom.
- `interatomic_force()`: Same as above but returns the magnitude of the Lennard-Jones force.
- `cancelled_linear_momentum()`: Takes the velocity of each atom, and adjusts them to cancel out any overall linear momentum (drift).
- `temp_from_vel()`: Returns the temperature in Kelvin from the velocity of each atom.
- `scaled_velocities()`: Takes the velocity of each atom and a temperature, and returns the velocities scaled such that they match the temperature.

In order to extend this code to different interatomic potentials or atomic species, the functions `interatomic_potential()` and `interatomic_force()` would need to be edited, including the parameters inside.

### **Run the code cell below to load the function definitions**

Try to modify the values of the LJ parameters to see the effects of stronger/weaker vdW forces.   

In [ ]:
"""
Simple implementation of a molecular dynamics code for a Lennard-Jones system

Inspiration is taken from a Fortran code for liquid Argon written by Alex
Pacheco on 30/01/2014 and modified by Devis Di Tommaso on 08/02/2017.  The atoms
start at FCC positions, are assigned a random velocity of magnitudes fulfilling
an initial temperature requirement, and then evolve in a cubic periodic simulation
cell according to the velocity Verlet algorithm. A scaling thermostat is applied at each step.

The units are Angstrom, seconds and kJ/mol. Note that the kJ/mol energy is that of the total cell, not per atom.
03/03/2021 - Miguel Rivera

"""
import numpy as np
import time
from itertools import combinations

### Functions

def pos_to_xyz(pos,file_name):
    with open(file_name, "a") as file_out:
        file_out.write(str(len(pos)) + "\n\n")
        # for line in pos*1e10:
        for line in pos:
            file_out.write("Ar {:9.3f} {:9.3f} {:9.3f}\n".format(line[0], line[1], line[2]))
    return


def interatomic_potential(inter_dist):
    """
    Return the energy of the van der Waals interaction given an interatomic distance

    Parameters
    ----------
    inter_dist : float
        Distance between two atoms in Angstroms
    Returns
    -------
    ener : float
        Interaction energy in kJ / mol

    """
    epsilon = 0.99363  # depth of the well in kJ/mol
    sigma = 3.405  # LJ interaction = 0 distance in A
    ener = 4 * epsilon * ((sigma / inter_dist) ** 12 - (sigma / inter_dist) ** 6)

    return ener


def interatomic_force(inter_dist):
    """
    Return the force resulting from the interatomic van der Waals interaction

    Parameters
    ----------
    inter_dist : float
        Distance between two atoms in Angstroms
    Returns
    -------
    force : float
        Magnitude of the resulting force in kJ / (mol * Angstrom)

    """
    epsilon = 0.99363  # depth of the well in kJ/mol
    sigma = 3.405  # LJ interaction = 0 distance in A
    force = (
        48
        * epsilon
        * ((sigma ** 12) / (inter_dist ** 13) - 0.5 * (sigma ** 6) / (inter_dist ** 7))
    )
    return force


def cancelled_linear_momentum(vel):
    """
    Set the linear momentum of a velocity array to 0

    Parameters
    ----------
    vel : N x 3 numpy array
        Initial velocities
    Returns
    -------
    vel_cancelled : N x 3 numpy array
        Velocities after cancellation

    """
    # velocity of the centre of mass
    com_vel = np.sum(vel, axis=0)
    # remove COM velocity/number of atoms from all atoms
    vel_cancelled = vel - com_vel / len(vel)

    return vel_cancelled


def temp_from_vel(vel):
    """Return temperature in T from a velocity array in A/s"""
    # Angstrom**2 to meter**2 conversion
    prefactor = 1e-20
    temperature = (
        prefactor
        * np.sum(np.linalg.norm(vel, axis=1) ** 2)
        * atomic_mass
        / (3 * g_const * n_atoms)
    )
    return temperature


def scaled_velocities(vel, temperature):
    """
    Scale a velocity vector according to a certain temperature

    Parameters
    ----------
    vel : N x 3 numpy array
        Initial velocities
    temperature : float
        Temperature in K
    Returns
    -------
    vel_scaled : N x 3 numpy array
        Scaled velocities

    """
    temp_initial = temp_from_vel(vel)
    fscal = np.sqrt(temperature / temp_initial) 
    vel_scaled = vel * fscal 
    return vel_scaled, fscal

### Run the code cell below to prepare the model system of Ar atoms 

The *box* of the model system is created by replicating a minima FCC  unit cell corresponding to **solid Argon**. The number of replicas (the `n_cell`parameter) determines the number of atoms $N$ to be included in the model system and the box size. Other critical MD settings are also defined in this cell:
* time step
* number of (integration) steps
* temperature (well above the melting point, so we'll simulate a liquid).


In [ ]:
### Preparing the MD system

# parameters
n_cell = 2 # number of unit cells along one axis of the supercell
lattice_const = 5.256  # in A
box_side = n_cell * lattice_const  # side of the simulation box in A
n_steps = 5000 # number of simulation steps
temp = 87.81  # in K
time_step = 2e-15  # s
atomic_mass = 39.95  # in u
g_const = 8.31446261815324e-3  # gas constant in kJ / (mol * K)
pos_file = "trajectory.xyz"
report_file = "report.csv"

### Initial positions

# cubic lattice
lattice_vectors = np.array(
    [
        [lattice_const, 0.0, 0.0],
        [0.0, lattice_const, 0.0],
        [0.0, 0.0, lattice_const],
    ]
)

# one FCC cell
fcc_pos = (
    np.array([[0.0, 0.0, 0.0], [0.5, 0.5, 0.0], [0.0, 0.5, 0.5], [0.5, 0.0, 0.5]])
    * lattice_const
)

# supercell of FCC cells
pos_i = []
for i in range(n_cell):
    for j in range(n_cell):
        for k in range(n_cell):
            # generate a translation vector
            trans_mult = np.array([i, j, k])
            trans_vec = np.sum(lattice_vectors * trans_mult, axis=1)

            # append new cell
            new_cell = fcc_pos + trans_vec
            pos_i.extend(new_cell.tolist())

pos_i = np.array(pos_i)

n_atoms=len(pos_i)

print (f'n_atoms in the box={n_atoms} ')

### Run  the  code cell below to initialize molecular velocities using MB distribution  

In [ ]:
### Assign starting velocities
from scipy.stats import Normal 
a=np.sqrt(g_const*temp/(atomic_mass))  
# Calculate an array of random speed values distributed according to MB 1D PDF = Normal distribution 
prefactor=1E10 # Conversion  from m into Angstroms
vel_i=prefactor * Normal(sigma=a).sample( (n_atoms,3)  )  # Angstrom/s
# set linear momentum to 0
vel_i = cancelled_linear_momentum(vel_i)


### Run the code cell below to run the simulation 
The simulation may take a few minutes to complete ....

In [ ]:
# Simulation loop

# start with null acceleration
acc_i = np.zeros((n_atoms, 3))

# start a output files
with open(pos_file, "w") as fname:
    pos_to_xyz(pos_i,pos_file)

with open(report_file,"w") as fname:
    fname.write("Step, Temperature, Energy\n")

# timer
start_time = time.perf_counter()

# Print frequency
nfreq= int( n_steps * 0.05 )

# loop each MD step
for i in range(n_steps):
    # velocity Verlet algorithm position update
    pos_f = pos_i + vel_i * time_step + (acc_i * time_step ** 2) / 2

    # apply periodic boundary conditions
    pos_f = pos_f % box_side

    # interatomic distances
    ener = 0
    force = np.zeros((n_atoms, 3))

    ## Apply potentials to the positions using numpy vectorisation
    # all pair positions
    comb = np.array(list(combinations(pos_f, 2)))

    # distance vector between each pair
    dist_vecs = comb[:, 1, :] - comb[:, 0, :]

    # periodic boundary
    dist_vecs_pbc = (dist_vecs + box_side / 2) % box_side - box_side / 2

    # array of scalar distances
    distances = np.linalg.norm(dist_vecs_pbc,axis=1)

    # calculate energies
    ener = np.sum(interatomic_potential(distances))

    # array of vector force per pair
    force_by_pair = dist_vecs_pbc / distances[:,None] * np.array(interatomic_force(distances))[:,None]

    # reshape it into a n_atoms x n_atoms (x 3) upper matrix
    force_mat = np.zeros((n_atoms,n_atoms,3))
    mask = np.triu_indices(n_atoms,k=1)
    force_mat[mask] = force_by_pair

    # add up rows and take away columns for a total
    force = np.sum(force_mat,axis=0) - np.sum(force_mat,axis=1)

    ## New velocities form forces
    # F = ma with 1e20 for Angstrom**2 to meter**2
    acc_f = force * 1e20 / ( atomic_mass )   # kJ / (mol Angstrom)

    # velocity update
    vel_f = vel_i + (acc_f + acc_i) / 2 * time_step
    kener = (1/2) * atomic_mass * np.sum ( vel_f**2)*1e-20  # kJ/mol  

    # print out report
    if  int ( i / nfreq ) * nfreq  == i : 
        print(
        "Step : {:>4} Temperature: {:>7.3f} K Potential energy: {:>9.3f} kJ/mol  Kinetic energy: {:>9.3f} kJ/mol".format(
            i + 1, temp_from_vel(vel_f), ener, kener)
        )
        print(
        "   (per atom values)                 Potential energy: {:>9.3f} kJ/mol  Kinetic energy: {:>9.3f} kJ/mol".format(ener/n_atoms, kener/n_atoms)
        )
        with open(report_file,"a") as fname:
            fname.write("{},{},{},{}\n".format(i+1,temp_from_vel(vel_f),ener,kener))
        # print out new goemetry
        pos_to_xyz(pos_f, "trajectory.xyz")

    # Scaling velocities (every 50 time steps)
    if  int ( i / ( 50 ) ) * ( 50 )  == i : 
        # scale velocities according to temperature
        vel_f , fscal = scaled_velocities(vel_f, temp)
        # set linear momentum to 0
        vel_f = cancelled_linear_momentum(vel_f)
        print(f" velocities scaled by factor= {fscal}") 
        
    
   # set up for next step
    pos_i = pos_f
    vel_i = vel_f
    acc_i = acc_f

# timer
end_time = time.perf_counter()
print("Run time: {:>8.4f} s.".format(end_time - start_time))

### Run the code cell below to calculate the radial distribution function

The peaks of the rdf indicate the position of the concentric spherical coordination shell. Only the coordinates of the last frame are considered.

In [ ]:
def rdf(pos,dr):
    """
    Return a radial distribution

    Arguments
    ---------
    pos : N x 3 numpy array
        Cartesian coordinates
    dr : float
        Radius step size between 0 and box_side length in Angstrom
    Returns
    -------
    x_range : numpy array
        X coordinate of the beginning of each bin
    left_bin_edges : numpy array
        Normalised count of interatomic distances within the bin. The normalisation is by the volume of the shell.

    """
    x_range = np.arange(0, box_side+dr, dr)

    # all distances with no double counting
    comb = np.array(list(combinations(pos, 2)))
    dist_vecs = comb[:, 1, :] - comb[:, 0, :]
    dist_vecs_pbc = (dist_vecs + box_side / 2) % box_side - box_side / 2
    distances = np.linalg.norm(dist_vecs_pbc,axis=1)

    # sort counts in a histogram
    counts, tmp_range = np.histogram(distances, x_range)
    left_bin_edges = x_range[:-1]

    # normalize by number of distances and particle density
    norm_counts = 2*counts/(((left_bin_edges+dr)**3 - left_bin_edges**3)* 4*np.pi/3 * n_atoms**2/box_side**3)

    # only return first half of the data, the latter is ruined by PBC
    return left_bin_edges[:round(len(left_bin_edges)/2)], norm_counts[:round(len(norm_counts)/2)]

# calculate the radial distribution function
dr = 0.25
rdf_distance, radial_density = rdf(pos_f,dr)
with open("rdf.csv","w") as rdf_file:
    rdf_file.write("Distance (Å),Density\n")
    for i,j in zip(rdf_distance, radial_density):
        rdf_file.write("{},{}\n".format(i,j))
        
# plot the result
import matplotlib.pyplot as plt
plt.plot(rdf_distance, radial_density)
plt.xlabel('r (Angstroms)')
plt.ylabel('RDF')
plt.grid()
plt.savefig("rdf.png")